# ST-GNN Sequential Floating Feature Selection (SFFS)

In [2]:
import os
import boto3
import json
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import mlflow
import mlflow.pytorch
import torch
import random
from snowML.datapipe.utils import data_utils as du
from snowML.datapipe.utils import get_geos as gg

import importlib
import train_eval_pipeline_feature_selection as tefs
importlib.reload(tefs)
from train_eval_pipeline_feature_selection import train_model

/home/sagemaker-user/.conda/envs/myenv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [ ]:
# =============================================================================
# CONFIG: data, HUCs, features, hyperparameters, SFFS thresholds
# =============================================================================

# --- S3 / Data source ---
BUCKET_NAME = "snowml-shape"
MODEL_READY_BUCKET = "snowml-model-ready-stgnn"
AWS_REGION = "us-west-2"

# --- Region / HUCs ---
HUC_LEVEL = "12"
HUC8_IDS = [
    17020009, 17020011, 17110005, 17110006, 17110009,
    17010304, 17010302, 17060207, 17060208, 17030001, 17030002,
    17110008, 17030003, 17020010, 17110007,
]
EXCLUDED_HUCS_CA = [
    "171100050101", "171100050102", "171100050201", "171100050202", "171100050203",
    "171100050301", "171100050302", "171100050303", "171100050304", "171100050305",
    "171100050306", "171100050401",
]
WEIRD_HUCS = ['171100060305', '171100060401', '171100060404', '171100060405']
EXCLUDED_HUCS = EXCLUDED_HUCS_CA + WEIRD_HUCS

# Adjacency matrix features
ADJ_SIGMA_CUTOFF = 3
ADJ_ALPHA = 0.7

# Base feature set: precip, temp, elevation 
BASE_DYNAMIC = ["mean_pr", "mean_tair"]
BASE_STATIC_MR = ["Mean Elevation"]
BASE_STATIC_GEOM = []
BASE_STATIC_DERIVED = []

# --- Extra features ---
EXTRA_DYNAMIC = ["mean_swe_lag_30", "snow_cover", "mean_hum", "mean_srad", "mean_vs"] #no mean_swe_lag_7
EXTRA_STATIC_MR = ["slope", "Predominant Snow", "Mean Forest Cover"]
EXTRA_STATIC_DERIVED = ["mean_pr_djf", "mean_tair_djf"]
EXTRA_STATIC_GEOM = ["area"]

# --- All features ---
ALL_DYNAMIC = BASE_DYNAMIC + EXTRA_DYNAMIC
ALL_STATIC_GEOM = BASE_STATIC_GEOM + EXTRA_STATIC_GEOM
ALL_STATIC_MR = BASE_STATIC_MR + EXTRA_STATIC_MR
ALL_STATIC_DERIVED = BASE_STATIC_DERIVED + EXTRA_STATIC_DERIVED

# --- Train / test split ---
TRAIN_SIZE_FRACTION = 0.67 # 67% train period, 33% test period

# --- Training hyperparameters (passed to train_model) ---
NUM_EPOCHS = 20
BATCH_SIZE = 32
SEQ_LEN = 30
VAL_SPLIT = 0.2
EARLY_STOPPING = True
PATIENCE = 5
REL_THRESHOLD = 0.03
RANDOM_SEED = 41

# SFFS thresholds
TAU_ADD = 0.001
TAU_REMOVE = 0.001
MSE_TOL_MIN = 0.0001
MSE_TOL_FRAC = 0.05

# --- MLflow ---
MLFLOW_TRACKING_URI = "arn:aws:sagemaker:us-west-2:677276086662:mlflow-tracking-server/wi26-snowml-mlflow"
#EXPERIMENT_NAME = "stgnn_swe_sffs"
#EXPERIMENT_NAME = "stgnn_swe_sffs_nolag"
#EXPERIMENT_NAME = "stgnn_swe_sffs_nobase"
#EXPERIMENT_NAME = "stgnn_swe_sffs_nobaseNlag"
#EXPERIMENT_NAME = "stgnn_swe_sffs_baselag30"
EXPERIMENT_NAME = "stgnn_swe_sffs_WbaseWOlag7"

In [59]:
s3_client = boto3.client("s3", region_name=AWS_REGION)

## Load GeoJSON and model-ready data

In [60]:
def get_geo_jsons(huc_ids, huc_level, bucket_nm="snowml-shape"):
    gdf = []
    for huc_id in huc_ids:
        temp = None
        f_out = f"Huc{huc_level}_in_{huc_id}.geojson"
        if du.isin_s3(bucket_nm, f_out):
            temp = du.s3_to_gdf(bucket_nm, f_out)
        else:
            try:
                temp = gg.get_geos(huc_id, huc_level, s3_save=True, bucket_nm=bucket_nm)
            except Exception as e:
                print(f"Error getting geos for {huc_id}: {e}")
        if temp is not None:
            temp['huc8'] = huc_id
            gdf.append(temp)
    return gpd.GeoDataFrame(pd.concat(gdf, ignore_index=True))


def get_model_ready_data(huc_ids, bucket_nm=MODEL_READY_BUCKET):
    huc_dfs = []
    for huc_id in huc_ids:
        f_out = f"model_ready_huc{huc_id}.csv"
        df = du.s3_to_df(f_out, bucket_nm)
        df['huc_id'] = huc_id
        df['day'] = pd.to_datetime(df["day"])
        huc_dfs.append(df)
    return pd.concat(huc_dfs, ignore_index=True)


gdf = get_geo_jsons(HUC8_IDS, HUC_LEVEL, BUCKET_NAME)
gdf = gdf[~gdf['huc_id'].isin(EXCLUDED_HUCS)]
huc12_ids = gdf["huc_id"]

huc_dfs = get_model_ready_data(huc12_ids)
huc_dfs = huc_dfs.dropna(subset=['mean_swe_lag_30']).copy()
print(f"gdf: {len(gdf)} HUC12s, huc_dfs: {len(huc_dfs)} rows")

gdf: 535 HUC12s, huc_dfs: 7604490 rows


## Time series tensor and adjacency

In [61]:
def get_time_series_data(huc_dfs, huc_ids, feature_names=BASE_DYNAMIC):
    huc_groups = huc_dfs.groupby('huc_id')['day'].apply(lambda x: set(x.astype(str)))
    common_timestamps = set.intersection(*huc_groups)
    master_index = sorted(common_timestamps)
    all_data = []
    target_data = []
    for huc in huc_ids:
        df = huc_dfs[huc_dfs['huc_id'] == huc].copy()
        df['day'] = df['day'].astype(str)
        df = df.set_index('day').reindex(master_index)
        features = df[feature_names].values.T.astype(float)
        all_data.append(features)
        target = df['mean_swe'].values.astype(float)
        target_data.append(target)
    dynamic_features = torch.tensor(np.array(all_data), dtype=torch.float32)
    target_tensor = torch.tensor(np.array(target_data), dtype=torch.float32)
    return dynamic_features, target_tensor


def normalize(arr):
    arr = np.array(arr)
    mu = np.nanmean(arr)
    sigma = np.nanstd(arr)
    if sigma == 0 or not np.isfinite(sigma):
        return np.zeros_like(arr)
    return (arr - mu) / sigma

In [62]:
def build_huc12_adjacency_hybrid(gdf, epsg_project=32610, alpha=ADJ_ALPHA, prox_cutoff_sigma=ADJ_SIGMA_CUTOFF,
                                 add_self_loops=True, normalize_adj=True, eps=1e-12):
    g = gdf[["huc_id", "geometry"]].copy().to_crs(epsg=epsg_project).reset_index(drop=True)
    N = len(g)
    geoms = g.geometry.values
    perim = np.array([geom.length for geom in geoms], dtype=float)
    area = np.array([geom.area for geom in geoms], dtype=float)
    centroids = g.geometry.centroid
    coords = np.array([[c.x, c.y] for c in centroids], dtype=float)
    diff = coords[:, None, :] - coords[None, :, :]
    d_mat = np.linalg.norm(diff, axis=2)
    np.fill_diagonal(d_mat, 0.0)
    d_nonzero = d_mat[d_mat > 0]
    sigma = float(np.median(d_nonzero) + eps)
    cutoff = prox_cutoff_sigma * sigma
    w_prox = np.exp(-(d_mat ** 2) / (2.0 * sigma ** 2))
    w_prox[d_mat > cutoff] = 0.0
    np.fill_diagonal(w_prox, 0.0)
    w_shared = np.zeros((N, N), dtype=float)
    sindex = g.sindex
    for i in range(N):
        geom_i = geoms[i]
        b_i = geom_i.boundary
        cand = list(sindex.intersection(geom_i.bounds))
        for j in cand:
            if j <= i:
                continue
            geom_j = geoms[j]
            if not geom_i.touches(geom_j):
                continue
            shared = b_i.intersection(geom_j.boundary)
            L = float(shared.length)
            wij = float((2.0 * L) / (perim[i] + perim[j] + eps))
            w_shared[i, j] = wij
            w_shared[j, i] = wij
    A_raw = alpha * w_shared + (1.0 - alpha) * w_prox
    A_used = A_raw.copy()
    if add_self_loops:
        np.fill_diagonal(A_used, A_used.diagonal() + 1.0)
    if normalize_adj:
        d = A_used.sum(axis=1)
        D_inv_sqrt = np.diag(1.0 / np.sqrt(d + eps))
        A_used = D_inv_sqrt @ A_used @ D_inv_sqrt
    return A_raw, A_used, coords, area, sigma

In [63]:
def create_static_features(gdf, huc_dfs, huc_ids, areas=None, static_from_geometry=None,
                           static_from_model_ready=None, derived_static=None, pr_tair_df=None,
                           return_group_index=False):
    if static_from_geometry is None:
        static_from_geometry = []
    if static_from_model_ready is None:
        static_from_model_ready = ["Mean Elevation"]
    static_df = huc_dfs.drop_duplicates("huc_id").set_index("huc_id").reindex(huc_ids)
    feature_list = []
    group_index = {}
    col_start = 0
    if "area" in static_from_geometry:
        if areas is None:
            raise ValueError("areas required")
        feature_list.append(normalize(areas)[:, None])
        group_index["area"] = [col_start]
        col_start += 1
    for col in static_from_model_ready:
        if col not in static_df.columns:
            raise ValueError(f"Static column '{col}' not in model-ready data.")
        s = static_df[col]
        if pd.api.types.is_numeric_dtype(s):
            feature_list.append(normalize(s.values)[:, None])
            group_index[col] = [col_start]
            col_start += 1
        else:
            onehot = pd.get_dummies(s.fillna("Unknown"), prefix=col.replace(" ", "_"))
            feature_list.append(onehot.values)
            group_index[col] = list(range(col_start, col_start + onehot.shape[1]))
            col_start += onehot.shape[1]
    if derived_static and pr_tair_df is not None:
        if "mean_pr_djf" in derived_static or "mean_tair_djf" in derived_static:
            pr_tair_djf_means = pr_tair_df.groupby("huc_id")[["mean_pr", "mean_tair"]].mean().reindex(huc_ids)
            raw_pr = pr_tair_djf_means["mean_pr"].to_numpy()
            raw_tair = pr_tair_djf_means["mean_tair"].to_numpy()
            if "mean_pr_djf" in derived_static:
                feature_list.append(normalize(raw_pr)[:, None])
                group_index["mean_pr_djf"] = [col_start]
                col_start += 1
            if "mean_tair_djf" in derived_static:
                feature_list.append(normalize(raw_tair)[:, None])
                group_index["mean_tair_djf"] = [col_start]
                col_start += 1
    if not feature_list:
        raise ValueError("At least one static feature required.")
    features = np.concatenate(feature_list, axis=1)
    static_features = torch.tensor(features, dtype=torch.float32)
    if return_group_index:
        return static_features, group_index
    return static_features

In [ ]:
# Build full dynamic and static tensors (all features)
dynamic_full, target_tensor = get_time_series_data(huc_dfs, huc12_ids, feature_names=ALL_DYNAMIC)
num_timesteps = dynamic_full.shape[2]
train_idx = int(num_timesteps * TRAIN_SIZE_FRACTION)
train_only_idx = int(train_idx * (1 - VAL_SPLIT))

dyn_train = dynamic_full[:, :, :train_idx].numpy()
dyn_mean = np.nanmean(dyn_train, axis=(0, 2))
dyn_std = np.nanstd(dyn_train, axis=(0, 2))
dyn_std = np.where(np.isfinite(dyn_std) & (dyn_std > 0), dyn_std, 1.0)
dynamic_full = (dynamic_full - torch.tensor(dyn_mean, dtype=torch.float32)[None, :, None]) / torch.tensor(dyn_std, dtype=torch.float32)[None, :, None]

dyn_index = {name: i for i, name in enumerate(ALL_DYNAMIC)}

gdf_huc12 = gdf.copy()
A_raw, A_norm, coords, areas, sigma = build_huc12_adjacency_hybrid(gdf_huc12)
adj_matrix = torch.tensor(A_norm, dtype=torch.float32)

pr_tair = huc_dfs[['huc_id', 'day', 'mean_pr', 'mean_tair']].copy()
pr_tair['day'] = pd.to_datetime(pr_tair['day'])
last_train_day = pr_tair['day'].drop_duplicates().sort_values().iloc[train_only_idx - 1]
pr_tair_djf = pr_tair[(pr_tair['day'] <= last_train_day) & (pr_tair['day'].dt.month.isin([12, 1, 2]))]

static_full, static_group_index = create_static_features(
    gdf_huc12, huc_dfs, huc12_ids, areas=areas,
    static_from_geometry=ALL_STATIC_GEOM,
    static_from_model_ready=ALL_STATIC_MR,
    derived_static=ALL_STATIC_DERIVED,
    pr_tair_df=pr_tair_djf,
    return_group_index=True,
)

print(f"dynamic_full {dynamic_full.shape}, static_full {static_full.shape}, train_idx={train_idx}")

## SFFS: Base set and candidate pool 

Base = from config (BASE_DYNAMIC + STATIC). Candidate pool = all features (ALL_DYNAMIC + static). Step 1 uses pool \ F_current. Each name maps to either dyn_index (dynamic) or static_group_index (static).

In [ ]:
# Base feature set: from config (precip, temp, elevation, etc.)
BASE_FEATURES = (BASE_DYNAMIC + BASE_STATIC_MR + BASE_STATIC_DERIVED + BASE_STATIC_GEOM)

# Candidate pool: all features; Step 1 uses (pool - F_current)
CANDIDATE_POOL = (ALL_DYNAMIC + ALL_STATIC_DERIVED + ALL_STATIC_GEOM + ALL_STATIC_MR)


def get_dyn_static_for_names(feature_names):
    """Given a list of feature names, return dynamic indices and static indices for slicing."""
    dyn_names = [n for n in feature_names if n in dyn_index]
    static_names = [n for n in feature_names if n in static_group_index]
    dyn_indices = [dyn_index[n] for n in dyn_names]
    static_indices = sum([static_group_index[n] for n in static_names], [])
    return dyn_indices, static_indices


def get_tensors_for_features(feature_names):
    """Return (train_dyn, train_tgt, static_feat) for the given feature set (train period only)."""
    dyn_indices, static_indices = get_dyn_static_for_names(feature_names)
    if not dyn_indices:
        raise ValueError("At least one dynamic feature required.")
    train_dyn = dynamic_full[:, dyn_indices, :train_idx]
    train_tgt = target_tensor[:, :train_idx]
    static_feat = static_full[:, static_indices] if static_indices else None
    if static_feat is not None and static_feat.shape[1] == 0:
        static_feat = None
    return train_dyn, train_tgt, static_feat

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def _safe_run_name(s):
    """MLflow run names: replace spaces for readability."""
    return (s or "").replace(" ", "_")


def train_and_eval(
    feature_names,
    run_name,
    sffs_step,
    current_features,
    added_feature=None,
    removed_feature=None,
    floating_round=None,
):
    """Train model, log params to MLflow; do not end run. Returns (model, val_kge, val_mse). Caller logs model and ends run."""
    train_dyn, train_tgt, static_feat = get_tensors_for_features(feature_names)
    set_seed(RANDOM_SEED)
    mlflow.start_run(run_name=run_name)
    params = {
        "sffs_step": sffs_step,
        "current_features": ",".join(current_features),
        "train_size_fraction": TRAIN_SIZE_FRACTION,
        "num_epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE,
        "seq_len": SEQ_LEN,
        "val_split": VAL_SPLIT,
        "early_stopping": EARLY_STOPPING,
        "patience": PATIENCE,
        "rel_threshold": REL_THRESHOLD,
        "tau_add": TAU_ADD,
        "tau_remove": TAU_REMOVE,
        "mse_min_tolerence":MSE_TOL_MIN,
        "mse_tolerence_fraction":MSE_TOL_FRAC,
    }
    if added_feature is not None:
        params["added_feature"] = added_feature
    if removed_feature is not None:
        params["removed_feature"] = removed_feature
    if floating_round is not None:
        params["floating_round"] = int(floating_round)
    mlflow.log_params(params)

    model, best_val = train_model(
        dynamic_features=train_dyn,
        target_tensor=train_tgt,
        adj_matrix=adj_matrix.numpy() if isinstance(adj_matrix, torch.Tensor) else adj_matrix,
        static_features=static_feat,
        num_epochs=NUM_EPOCHS,
        batch_size=BATCH_SIZE,
        seq_len=SEQ_LEN,
        val_split=VAL_SPLIT,
        early_stopping=EARLY_STOPPING,
        patience=PATIENCE,
        rel_threshold=REL_THRESHOLD,
        plot=False,
    )
    val_kge = best_val["val_kge"]
    val_mse = best_val["val_mse"]
    if np.isnan(val_kge):
        val_kge = -np.inf
    return model, float(val_kge), float(val_mse)

## SFFS accept rules and mse tolerence

- **Add**: M' better than M iff ΔKGE ≥ tau_add AND ΔMSE ≤ mse_tol(M).  
- **Remove**: M' better than M iff ΔKGE ≥ tau_remove AND ΔMSE ≤ mse_tol(M).  
- mse_tol(M) = max(0.0001, 0.05 * val_mse(M)).  
- ΔKGE = val_kge(M') - val_kge(M), ΔMSE = val_mse(M') - val_mse(M).

In [ ]:
def mse_tol(val_mse_current):
    return max(MSE_TOL_MIN, MSE_TOL_FRAC * val_mse_current)


def accept_add(val_kge_new, val_mse_new, val_kge_current, val_mse_current):
    """M' (new) is better than M (current) for addition."""
    dkge = val_kge_new - val_kge_current
    dmse = val_mse_new - val_mse_current
    return dkge >= TAU_ADD and dmse <= mse_tol(val_mse_current)


def accept_remove(val_kge_new, val_mse_new, val_kge_current, val_mse_current):
    """M' (after removal) is better than M (current) for removal."""
    dkge = val_kge_new - val_kge_current
    dmse = val_mse_new - val_mse_current
    return dkge >= TAU_REMOVE and dmse <= mse_tol(val_mse_current)

## Sequential Floating Feature Selection (algorithm)

- **Step 0**: Train base model. Record KGE_current, MSE_current.
- **Step 1 (Forward)**: For every candidate f not in F_current, train F_current ∪ {f}. Find best that satisfies add rule. If none → STOP. Else add best → F_current, go to Step 2.
- **Step 2 (Floating removal)**: For every g in F_current, train F_current \ {g}. If any satisfies remove rule, remove the best, update F_current, **repeat Step 2**. If no removal satisfies → go back to Step 1 (until Step 1 stops).

In [ ]:
print(BASE_FEATURES, CANDIDATE_POOL)

In [ ]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)
if mlflow.active_run() is not None:
    mlflow.end_run()

F_current = list(BASE_FEATURES)
history = []
runs_log = []

if F_current:
    # Step 0: Train base (log model)
    print("Step 0: Training base model", F_current)
    model, KGE_current, MSE_current = train_and_eval(
        F_current,
        run_name="sffs_base",
        sffs_step="base",
        current_features=F_current,
    )
    mlflow.pytorch.log_model(model, "model")
    mlflow.end_run()
    runs_log.append({"run_name": "sffs_base", "best_val_kge": KGE_current, "best_val_mse": MSE_current})
    history.append({"step": 0, "action": "base", "F_current": list(F_current), "val_kge": KGE_current, "val_mse": MSE_current})
    print(f"  val_kge={KGE_current:.6f}, val_mse={MSE_current:.6f}")
else:
    # Step 0 (no base): pick first feature = dynamic with max val_kge
    dynamic_candidates = [f for f in CANDIDATE_POOL if f in dyn_index]
    if not dynamic_candidates:
        raise ValueError("BASE_FEATURES is empty and no dynamic feature in CANDIDATE_POOL. Need at least one dynamic to start.")
    print("Step 0: No base; picking first feature (dynamic with max val_kge)", dynamic_candidates)
    best_kge, best_mse, best_f = -np.inf, np.inf, None
    for f in dynamic_candidates:
        model, kge_new, mse_new = train_and_eval(
            [f],
            run_name=f"sffs_first_try_{_safe_run_name(f)}",
            sffs_step="first_feature",
            current_features=[],
            added_feature=f,
        )
        runs_log.append({"run_name": f"sffs_first_try_{_safe_run_name(f)}", "best_val_kge": kge_new, "best_val_mse": mse_new})
        if kge_new > best_kge:
            best_kge, best_mse, best_f = kge_new, mse_new, f
            mlflow.pytorch.log_model(model, "model")
        mlflow.end_run()
    F_current = [best_f]
    KGE_current, MSE_current = best_kge, best_mse
    history.append({"step": 0, "action": "first_feature", "added": best_f, "F_current": list(F_current), "val_kge": KGE_current, "val_mse": MSE_current})
    print(f"  picked {best_f}. F_current={F_current}, val_kge={KGE_current:.6f}, val_mse={MSE_current:.6f}")

while True:
    # Step 1: Forward
    candidates = [f for f in CANDIDATE_POOL if f not in F_current]
    if not candidates:
        print("No more candidates. SFFS done.")
        break

    best_kge, best_mse, best_f = -np.inf, np.inf, None
    for f in candidates:
        F_try = F_current + [f]
        model, kge_new, mse_new = train_and_eval(
            F_try,
            run_name=f"sffs_fwd_try_add_{_safe_run_name(f)}",
            sffs_step="forward",
            current_features=F_current,
            added_feature=f,
        )
        runs_log.append({"run_name": f"sffs_fwd_try_add_{_safe_run_name(f)}", "best_val_kge": kge_new, "best_val_mse": mse_new})
        if accept_add(kge_new, mse_new, KGE_current, MSE_current):
            if kge_new > best_kge:
                best_kge, best_mse, best_f = kge_new, mse_new, f
                mlflow.pytorch.log_model(model, "model")
        mlflow.end_run()

    if best_f is None:
        print("Step 1: No feature satisfies add rule. Final model:", F_current)
        break

    F_current = F_current + [best_f]
    KGE_current, MSE_current = best_kge, best_mse
    history.append({"step": "forward", "action": "add", "added": best_f, "F_current": list(F_current), "val_kge": KGE_current, "val_mse": MSE_current})
    print(f"Step 1: Added {best_f}. F_current={F_current}, val_kge={KGE_current:.6f}, val_mse={MSE_current:.6f}")

    # Step 2: Floating removal (repeat until no removal satisfies)
    floating_round = 0
    while True:
        removal_candidates = list(F_current)
        best_rem_kge, best_rem_mse, best_rem_g = -np.inf, np.inf, None
        floating_round += 1
        for g in removal_candidates:
            F_try = [x for x in F_current if x != g]
            if not F_try:
                continue
            dyn_indices, _ = get_dyn_static_for_names(F_try)
            if not dyn_indices:
                continue
            model, kge_new, mse_new = train_and_eval(
                F_try,
                run_name=f"sffs_flt_round{floating_round}_try_remove_{_safe_run_name(g)}",
                sffs_step="floating",
                current_features=F_current,
                removed_feature=g,
                floating_round=floating_round,
            )
            runs_log.append({"run_name": f"sffs_flt_round{floating_round}_try_remove_{_safe_run_name(g)}", "best_val_kge": kge_new, "best_val_mse": mse_new})
            if accept_remove(kge_new, mse_new, KGE_current, MSE_current):
                if kge_new > best_rem_kge:
                    best_rem_kge, best_rem_mse, best_rem_g = kge_new, mse_new, g
                    mlflow.pytorch.log_model(model, "model")
            mlflow.end_run()

        if best_rem_g is None:
            break

        F_current = [x for x in F_current if x != best_rem_g]
        KGE_current, MSE_current = best_rem_kge, best_rem_mse
        history.append({"step": "floating", "action": "remove", "removed": best_rem_g, "F_current": list(F_current), "val_kge": KGE_current, "val_mse": MSE_current})
        print(f"Step 2: Removed {best_rem_g}. F_current={F_current}, val_kge={KGE_current:.6f}, val_mse={MSE_current:.6f}")

print("\nFinal selected features:", F_current)
print(f"Final val_kge={KGE_current:.6f}, val_mse={MSE_current:.6f}")

Epoch 5/20, Train - MSE: 0.0186, KGE: 0.7985, Val   - MSE: 0.0142, KGE: 0.8220


In [70]:
df_sffs_history = pd.DataFrame(history)
df_sffs_history

,step,action,F_current,val_kge,val_mse,added
0,0,base,"[mean_pr, mean_tair, Mean Elevation]",0.882715,0.009228,NaN
1,forward,add,"[mean_pr, mean_tair, Mean Elevation, mean_swe_...",0.976451,0.001136,mean_swe_lag_30


In [71]:
print("\nFinal selected features:", F_current)
print(f"Final val_kge={KGE_current:.6f}, val_mse={MSE_current:.6f}")


Final selected features: ['mean_pr', 'mean_tair', 'Mean Elevation', 'mean_swe_lag_30']
Final val_kge=0.976451, val_mse=0.001136


In [72]:
# Per-run summary: run_name, best_val_kge, best_val_mse (every train_and_eval)
df_runs = pd.DataFrame(runs_log)
df_runs

,run_name,best_val_kge,best_val_mse
0,sffs_base,0.882715,0.009228
1,sffs_fwd_try_add_mean_swe_lag_30,0.976451,0.001136
2,sffs_fwd_try_add_snow_cover,0.863381,0.007377
3,sffs_fwd_try_add_mean_hum,0.847818,0.012554
4,sffs_fwd_try_add_mean_srad,0.923156,0.007484
5,sffs_fwd_try_add_mean_vs,0.895798,0.009592
6,sffs_fwd_try_add_mean_pr_djf,0.822016,0.014196
7,sffs_fwd_try_add_mean_tair_djf,0.822016,0.014196
8,sffs_fwd_try_add_area,0.822016,0.014196
9,sffs_fwd_try_add_slope,0.822016,0.014196


## Retrieve Experiment Results from MLflow

This section pulls the results of a previously completed experiment directly from the MLflow tracking server.

The code queries MLflow for all runs associated with the specified experiment name and returns key metrics and parameters (e.g., added/removed features, validation MSE, and validation KGE).

Because the results are stored in MLflow, **the experiment does not need to be rerun**. This allows you to quickly inspect, analyze, and compare past runs without retraining the model.

To retrieve results for a different experiment, simply update the `EXPERIMENT_NAME` variable below.

**Note:** The runs in the resulting DataFrame (`df_runs`) are listed in reverse chronological order.
The **earliest run appears at the bottom**, and the **most recent run appears at the top**.
When interpreting the feature selection process, start from the **bottom of the DataFrame and read upward** to follow the order in which runs occurred.

In [6]:
import mlflow
from mlflow.tracking import MlflowClient
import pandas as pd

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient()

# Get experiment ID
EXPERIMENT_NAME = "stgnn_swe_sffs" # Update the experiment name below to retrieve results for that specific experiment
exp = client.get_experiment_by_name(EXPERIMENT_NAME)
exp_id = exp.experiment_id

# Pull all runs (you can add filter_string if you want only SFFS runs)
runs = client.search_runs(
    [exp_id],
    filter_string="",  # e.g. 'params.sffs_step = "forward"' if you want only forwards
    max_results=10000,
)

rows = []
for r in runs:
    rows.append({
        "run_name": r.info.run_name,
        "sffs_step": r.data.params.get("sffs_step"),
        "added_feature": r.data.params.get("added_feature"),
        "removed_feature": r.data.params.get("removed_feature"),
        "best_val_mse": r.data.metrics.get("best_val_mse"),
        "best_val_kge": r.data.metrics.get("best_val_kge"),
    })

df_runs = pd.DataFrame(rows)
df_runs

,run_name,sffs_step,added_feature,removed_feature,best_val_mse,best_val_kge
0,sffs_fwd_try_add_Mean_Forest_Cover,forward,Mean Forest Cover,None,0.000506,0.990615
1,sffs_fwd_try_add_Predominant_Snow,forward,Predominant Snow,None,0.000784,0.940542
2,sffs_fwd_try_add_slope,forward,slope,None,0.000506,0.990615
3,sffs_fwd_try_add_area,forward,area,None,0.000506,0.990615
4,sffs_fwd_try_add_mean_tair_djf,forward,mean_tair_djf,None,0.000506,0.990615
5,sffs_fwd_try_add_mean_pr_djf,forward,mean_pr_djf,None,0.000506,0.990615
6,sffs_fwd_try_add_mean_swe_lag_30,forward,mean_swe_lag_30,None,0.000474,0.963817
7,sffs_fwd_try_add_mean_vs,forward,mean_vs,None,0.000606,0.985138
8,sffs_fwd_try_add_mean_srad,forward,mean_srad,None,0.000487,0.986883
9,sffs_fwd_try_add_mean_hum,forward,mean_hum,None,0.000465,0.979964
